# NB03: Data Analysis

## Purpose

NB02 produced three tidy tables from the 500-page TMDB "popular movies" pull:
`movies.csv` (one row per film), `genres.csv` (id-to-name lookup), and
`movies_with_genres.csv` (one row per movie-genre pairing). This notebook uses
those tables to explore the question: **Do Popularity and Audience Ratings Move Together Across Film Genres?**

This notebook does not touch the API or repeat any cleaning from NB02 — it
only groups, aggregates, and charts the data NB02 already prepared.

In [6]:
import kaleido
import pandas as pd
import plotly.express as px

In [7]:
movies_df = pd.read_csv("../data/processed/movies.csv")
genres_df = pd.read_csv("../data/processed/genres.csv")
movies_with_genres_df = pd.read_csv("../data/processed/movies_with_genres.csv")

## The data this notebook uses

`movies_with_genres_df` is the main table: one row per movie-genre pairing, so
a film with three genres contributes three rows. This is the correct shape for
genre-level grouping, but means any `len()` on it counts movie-genre pairs, not
films — `n_movies` in the tables below uses `.nunique("id")` specifically to
avoid that trap.

Two things carried over from NB02 shape how I read the results here:

- **This is popularity-pre-filtered data.** The source, TMDB's `/movie/popular`
  endpoint, only returns films that are already popular. So this analysis
  cannot say anything about whether obscure or low-visibility films behave the
  same way — only how popularity and rating relate *within* the pool of films
  that already have some audience attention.
- **`vote_average` is an audience score, not a critics' score.** It is the
  average rating left by TMDB's own registered users, not a professional
  review aggregator like Metacritic or Rotten Tomatoes. I use "audience
  rating" throughout this notebook and the public page for that reason.

### Chart 1 — Bar chart, average popularity by genre

**Why a bar chart.** The question here is a ranking across a small number of
categories (19 genres), and a bar chart is the clearest way to compare one
number per category. Colour encodes `avg_popularity` again so the ranking is
visible at a glance, and `n_movies` is available on hover so a genre's rank
can be read alongside how much data supports it.

In [8]:
genre_stats = (movies_with_genres_df.groupby("genre").agg(n_movies=("id","nunique"), avg_popularity=("popularity","mean")) ).reset_index().sort_values("avg_popularity", ascending=False)

fig1 = px.bar(genre_stats, x="genre", y="avg_popularity",
             hover_data=["n_movies"], color="avg_popularity",color_continuous_scale="Tealgrn")
fig1.update_layout(title="Sci-Fi and Adventure films draw far more attention than Drama, despite being rarer")
fig1.write_html("../docs/charts/finding1_average_popularity_by_genre.html", include_plotlyjs="cdn", full_html=True)
fig1.show()

**Finding 1: attention is concentrated in a handful of genres, and it doesn't
track how often a genre appears.**

Science Fiction (16,78), Adventure (15,29) and Fantasy (13,07) draw far more
average popularity than the rest. Drama, despite being the single most common
genre in the dataset (2427 films), sits near the bottom (7.93) — only Documentary is lower (5.31). The genre made most often is not
the genre that draws the most attention.

### Chart 2 — Scatter plot, popularity vs. vote average by genre

**Why a scatter plot.** Chart 1 shows popularity; this chart adds a second
variable (audience rating) so the two can be compared directly, one point per
genre. Point size encodes `n_movies`, so a genre resting on very few films
(and therefore a less stable average) is visually distinguishable from one
backed by hundreds.

In [9]:
fig2 = px.scatter(genre_stats.merge(
                    movies_with_genres_df.groupby("genre")["vote_average"].mean().reset_index(),
                    on="genre"),
                  x="avg_popularity", y="vote_average", size="n_movies",
                  text="genre", hover_name="genre",color="genre")
fig2.update_layout(title="Popularity and critical rating don't move together across genres")
fig2.write_html("../docs/charts/finding2_popularityvsvote_by_decade.html", include_plotlyjs="cdn", full_html=True)
fig2.show()

**Finding 2: popularity and audience rating are largely separate.**

 Popularity and rating do not move together. Music has the highest average rating (7,36) despite low popularity, while Science Fiction and Adventure — the two most popular genres — sit in the middle of the rating range (6.64 and 6.65). Horror is the clearest outlier: it is fairly popular (10,52) but has the lowest average rating among major genres (6.25). Drama, the largest genre by count (2426 movies), has an average, unremarkable rating (7.96).

### Chart 3 — Line chart, genre share of top films by decade

**Why a line chart.** This chart tracks change over time, which a line makes
directly readable — a bar chart per decade would need six bars side by side
per period and hide the trend itself. I restrict to the top 6 genres by
frequency and to decades from 1970 onward, where the dataset has enough films
per decade for a share to be meaningful (earlier decades have very few TMDB
"popular" entries and would produce noisy, unreliable percentages).

In [10]:
top_genres = movies_with_genres_df["genre"].value_counts().head(6).index.tolist()  # .index extracts just the genre names from value_counts(), not the counts
sub = movies_with_genres_df[
    movies_with_genres_df["genre"].isin(top_genres)  # .isin() returns True/False for each row, checking if its genre is in top_genres
    & (movies_with_genres_df["decade"] >= 1970)
]
counts = (
    sub.groupby(["decade", "genre"])
       .size()
       .rename("n")  # names the unnamed count column from .size() so it can be referenced as "n"
       .reset_index()  # turns the multi-level (decade, genre) index back into regular columns
)
counts["share_pct"] = (
    100 * counts["n"]
    / counts.groupby("decade")["n"].transform("sum")  # .transform("sum") sums "n" per decade but keeps one row per original row, so it can be divided row-by-row
)
fig3 = px.line(counts, x="decade", y="share_pct", color="genre", markers=True)
fig3.update_layout(title="Drama's share of top films has fallen from 39% to 26% since the 1970s, as Action and Thriller doubled")
fig3.write_html("../docs/charts/finding3_genre_share_by_decade.html", include_plotlyjs="cdn", full_html=True)
fig3.show()

**Finding 3: the popularity/rating gap lines up with a five-decade shift.**

 Drama's share has fallen steadily, from 39,14% of top films in the 1970s to 26,73% in the 2020s. Action has more than doubled its share over the same period, from 9.53% to 17,76%, and Thriller has followed a similar pattern (11,18% to 18.16%). Comedy has also declined somewhat (22,69% to 17,44%), while Adventure and Romance have stayed comparatively flat across the decades. The overall pattern points to action-oriented genres steadily displacing Drama and Comedy in the composition of popular films.